# Deep Learning para Detecção de Fraudes em Transações Financeiras com Criptomoedas

In [31]:
# Imports
import sklearn
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras import Input
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import warnings
warnings.filterwarnings('ignore')

In [32]:
# Carrega os dados

df = pd.read_csv('dataset.csv')

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 51 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Unnamed: 0                                            9841 non-null   int64  
 1   Index                                                 9841 non-null   int64  
 2   Address                                               9841 non-null   object 
 3   FLAG                                                  9841 non-null   int64  
 4   Avg min between sent tnx                              9841 non-null   float64
 5   Avg min between received tnx                          9841 non-null   float64
 6   Time Diff between first and last (Mins)               9841 non-null   float64
 7   Sent tnx                                              9841 non-null   int64  
 8   Received Tnx                                          9841

A variável FLAG indica transação fraudulenta 

In [34]:
# Variável alvo
df['FLAG'].value_counts()

FLAG
0    7662
1    2179
Name: count, dtype: int64

## Limpeza e organização dos dados

In [35]:
df.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [36]:
# Ajusta o nome para minúsculo
df.columns = [x.lower() for x in df.columns]

In [37]:
# Essas colunas não serão relevantes para análise
cols_to_drop = [' erc20 most sent token type',
                ' erc20_most_rec_token_type',
                'address',
                'index',
                'unnamed: 0']

In [38]:
# Seleciona os atributos filtrando as colunas que serão removidas e a variável alvo
atributos = [x for x in df.columns if (x != 'flag' and x not in cols_to_drop)]

In [39]:
valores_unicos = df.nunique()

Remove atributos com valores únicos

In [40]:
atributos = [x for x in atributos if x in valores_unicos.loc[(valores_unicos > 1)]]

In [41]:
df[atributos].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 38 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   avg min between sent tnx                              9841 non-null   float64
 1   avg min between received tnx                          9841 non-null   float64
 2   time diff between first and last (mins)               9841 non-null   float64
 3   sent tnx                                              9841 non-null   int64  
 4   received tnx                                          9841 non-null   int64  
 5   number of created contracts                           9841 non-null   int64  
 6   unique received from addresses                        9841 non-null   int64  
 7   unique sent to addresses                              9841 non-null   int64  
 8   min value received                                    9841

## Criação do Pipeline de Pré-Processamento

**BaseEstimator**: Esta é a classe base para todos os estimadores no scikit-learn. Um estimador é qualquer objeto que pode estimar alguns parâmetros baseados em um conjunto de dados. Por exemplo, um algoritmo de Machine Learning como uma regressão linear ou uma árvore de decisão é um estimador. A classe BaseEstimator fornece funcionalidades básicas, como métodos para configurar os parâmetros de um estimador e para obter esses parâmetros com get_params() e set_params(). 

**TransformerMixin**: Esta é uma classe mista (mixin) usada para adicionar funcionalidade de transformação a um estimador. No scikit-learn, um "transformador" é um tipo de estimador que pode transformar um conjunto de dados. Por exemplo, ele pode ser usado para normalizar ou padronizar dados, selecionar ou extrair características, etc. A classe TransformerMixin adiciona o método fit_transform(), que é um método conveniente que primeiro ajusta o transformador ao conjunto de dados (usando fit()) e então aplica a transformação ao mesmo conjunto de dados (usando transform()). Isso é útil para evitar ter que chamar fit() e transform() separadamente durante o pré-processamento dos dados.

A combinação dessas duas classes é frequentemente usada ao criar um estimador personalizado ou transformador com o scikit-learn. Ao herdar dessas classes, você garante que seu estimador personalizado se integre bem com outras funcionalidades de scikit-learn, como pipelines e outras ferramentas de modelagem.

In [42]:
# Definição de uma classe personalizada que herda de BaseEstimator e TransformerMixin
class PipeSteps(BaseEstimator, TransformerMixin):

    # Método construtor para inicializar a classe com uma lista de colunas
    def __init__(self, columns=[]):
        
        # Atribuição do argumento columns ao atributo de instância self.columns
        self.columns = columns

    # Método fit usado para ajustar (treinar) a transformação nos dados de treinamento
    def fit(self, X, y = None):
        
        # Retorna a própria instância, indicando que o método não faz modificações
        return self

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Retorna os dados sem modificações
        return X

In [43]:
# Definição de uma classe que herda de PipeSteps
class SelecionaColunas(PipeSteps):

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Seleciona e retorna apenas as colunas especificadas em self.columns
        return X[self.columns]

In [44]:
# Definição de uma classe que herda de PipeSteps
class PreencheDados(PipeSteps):

    # Método fit para ajustar a transformação nos dados de treinamento
    def fit(self, X, y = None):
        
        # Calcula a média de cada coluna especificada em self.columns e armazena no dicionário self.means
        self.means = { col: X[col].mean() for col in self.columns }
        
        # Retorna a própria instância, indicando que o método não faz modificações
        return self

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Itera sobre cada coluna especificada em self.columns
        for col in self.columns:
            
            # Preenche valores ausentes na coluna com a média calculada na fase de ajuste
            X[col] = X[col].fillna(self.means[col])
        
        # Retorna os dados transformados
        return X

In [45]:
# Definição de uma classe que herda de PipeSteps
class PadronizaDados(PipeSteps):

    # Método fit para ajustar o scaler nos dados de treinamento
    def fit(self, X, y = None):
        
        # Inicializa uma instância de StandardScaler para padronizar os dados
        self.scaler = StandardScaler()
        
        # Ajusta o scaler nas colunas especificadas em self.columns
        self.scaler.fit(X[self.columns])
        
        # Retorna a própria instância, indicando que o método não faz modificações
        return self

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Aplica a transformação de padronização nas colunas especificadas
        X[self.columns] = self.scaler.transform(X[self.columns])
        
        # Retorna os dados transformados
        return X

In [46]:
# Definição de uma classe que herda de PipeSteps
class ObtemDados(PipeSteps):

    # Método transform para transformar os dados de entrada
    def transform(self, X):
        
        # Faz uma cópia dos dados de entrada para evitar alterar os dados originais
        X = X.copy()
        
        # Retorna os valores do DataFrame como um array NumPy
        return X.values

Pipeline no scikit-learn é uma ferramenta que encadeia várias etapas de transformação e um estimador final em um único objeto. Isso permite que você configure um processo de Machine Learning que inclui etapas de pré-processamento (como normalização, padronização e extração de características) seguidas pela aplicação de um modelo de aprendizado de máquina. Cada etapa da pipeline é representada por uma tupla contendo um nome e um objeto de transformação ou modelo.

In [47]:
# Cria o pipeline
pipe_prepropcessamento = Pipeline([('feature_selection', SelecionaColunas(atributos)),
                                   ('fill_missing', PreencheDados(atributos)),
                                   ('standard_scaling', PadronizaDados(atributos)),
                                   ('returnValues', ObtemDados())])

In [49]:
# Variáveis de entrada
X = df[atributos]

# Variável de saída
y = df['flag']

In [50]:
y = to_categorical(y)

In [ ]:
# Divide os dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size = 0.30, random_state = 29)

In [52]:
# Padroniza os dados
X_treino = pipe_prepropcessamento.fit_transform(X_treino)
X_teste = pipe_prepropcessamento.transform(X_teste)